# 288_12_18 DEM Sizes

This notebook extracts DEM-derived decoding matrix sizes for representative bicycle-bivariate `288_12_18` test circuits.

- `XZ` means basis-filtered decoding, matching `xyz=False` in the older notebooks.
- `XYZ` means full unfiltered decoding, matching `xyz=True` in the older notebooks.
- This version checks one representative `memory_X` file and one representative `memory_Z` file, instead of scanning every available `p`.

In [1]:
from pathlib import Path

import pandas as pd
import stim

from relay_bp.stim.sinter.check_matrices import CheckMatrices
from relay_bp.stim.sinter.runner import _filter_detectors_by_basis as filter_detectors_by_basis


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "tests" / "testdata" / "bicycle_bivariate").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find repo root containing tests/testdata/bicycle_bivariate "
        f"starting from {start}."
    )


repo_root = find_repo_root(Path.cwd().resolve())
circuits_dir = repo_root / "tests" / "testdata" / "bicycle_bivariate"

print("repo_root =", repo_root)
print("circuits_dir =", circuits_dir)


repo_root = C:\Users\User\Documents\projects-git\relay_mother_folder\relay
circuits_dir = C:\Users\User\Documents\projects-git\relay_mother_folder\relay\tests\testdata\bicycle_bivariate


In [2]:
def parse_name_metadata(path: Path) -> dict:
    metadata = {"stim_path": str(path)}
    for part in path.stem.split(","):
        if "=" not in part:
            continue
        key, value = part.split("=", 1)
        metadata[key] = value
    if "error_rate" in metadata:
        metadata["p"] = float(metadata["error_rate"])
    return metadata


def select_representative_paths() -> list[Path]:
    selected = []
    for memory_kind in ["X", "Z"]:
        matches = [
            path
            for path in sorted(
                circuits_dir.glob(
                    f"circuit=bicycle_bivariate_288_12_18_memory_{memory_kind},*.stim"
                )
            )
            if "memory_choi_XZ" not in path.name
        ]
        if not matches:
            raise FileNotFoundError(
                f"No 288_12_18 memory_{memory_kind} circuit found in {circuits_dir}."
            )
        selected.append(matches[0])
    return selected


def collect_dem_sizes(paths: list[Path]) -> pd.DataFrame:
    rows = []
    for path in paths:
        metadata = parse_name_metadata(path)
        circuit_name = metadata["circuit"]
        basis = circuit_name[-1]

        circuit_xyz = stim.Circuit.from_file(path)
        dem_xyz = circuit_xyz.detector_error_model()
        matrices_xyz = CheckMatrices.from_dem(dem_xyz, prune_decided_errors=False)

        circuit_xz = filter_detectors_by_basis(circuit_xyz, basis)
        dem_xz = circuit_xz.detector_error_model()
        matrices_xz = CheckMatrices.from_dem(dem_xz, prune_decided_errors=False)

        rows.append(
            {
                "circuit": circuit_name,
                "basis": basis,
                "p": metadata["p"],
                "xz_rows": matrices_xz.check_matrix.shape[0],
                "xz_cols": matrices_xz.check_matrix.shape[1],
                "xyz_rows": matrices_xyz.check_matrix.shape[0],
                "xyz_cols": matrices_xyz.check_matrix.shape[1],
                "xz_check_nnz": int(matrices_xz.check_matrix.nnz),
                "xyz_check_nnz": int(matrices_xyz.check_matrix.nnz),
                "xz_obs_rows": matrices_xz.observables_matrix.shape[0],
                "xz_obs_cols": matrices_xz.observables_matrix.shape[1],
                "xyz_obs_rows": matrices_xyz.observables_matrix.shape[0],
                "xyz_obs_cols": matrices_xyz.observables_matrix.shape[1],
                "xz_detectors": dem_xz.num_detectors,
                "xyz_detectors": dem_xyz.num_detectors,
                "xz_observables": dem_xz.num_observables,
                "xyz_observables": dem_xyz.num_observables,
                "stim_path": str(path),
            }
        )

    return pd.DataFrame(rows).sort_values(["basis", "p"]).reset_index(drop=True)


selected_paths = select_representative_paths()
print("Selected representative files:")
for path in selected_paths:
    print(" -", path.name)

sizes_df = collect_dem_sizes(selected_paths)
display(
    sizes_df[
        [
            "circuit",
            "basis",
            "p",
            "xz_rows",
            "xz_cols",
            "xyz_rows",
            "xyz_cols",
            "xz_check_nnz",
            "xyz_check_nnz",
            "stim_path",
        ]
    ]
)


Selected representative files:
 - circuit=bicycle_bivariate_288_12_18_memory_X,distance=18,rounds=18,error_rate=0.001,noise_model=uniform_circuit,basis=CX,A=x^3+y^2+y^7,B=y^3+x+x^2.stim
 - circuit=bicycle_bivariate_288_12_18_memory_Z,distance=18,rounds=18,error_rate=0.001,noise_model=uniform_circuit,basis=CX,A=x^3+y^2+y^7,B=y^3+x+x^2.stim


,circuit,basis,p,xz_rows,xz_cols,xyz_rows,xyz_cols,xz_check_nnz,xyz_check_nnz,stim_path
0,bicycle_bivariate_288_12_18_memory_X,X,0.001,2736,26208,5184,206496,91584,1201104,C:\Users\User\Documents\projects-git\relay_mot...
1,bicycle_bivariate_288_12_18_memory_Z,Z,0.001,2736,26208,5184,206352,91584,1200816,C:\Users\User\Documents\projects-git\relay_mot...


In [3]:
summary_df = sizes_df.copy()
summary_df["xz_check_shape"] = summary_df.apply(
    lambda row: f"{row['xz_rows']} x {row['xz_cols']}", axis=1
)
summary_df["xyz_check_shape"] = summary_df.apply(
    lambda row: f"{row['xyz_rows']} x {row['xyz_cols']}", axis=1
)

display(
    summary_df[
        [
            "circuit",
            "basis",
            "p",
            "xz_check_shape",
            "xyz_check_shape",
            "xz_check_nnz",
            "xyz_check_nnz",
            "xz_detectors",
            "xyz_detectors",
            "xz_observables",
            "xyz_observables",
        ]
    ]
)


,circuit,basis,p,xz_check_shape,xyz_check_shape,xz_check_nnz,xyz_check_nnz,xz_detectors,xyz_detectors,xz_observables,xyz_observables
0,bicycle_bivariate_288_12_18_memory_X,X,0.001,2736 x 26208,5184 x 206496,91584,1201104,2736,5184,12,12
1,bicycle_bivariate_288_12_18_memory_Z,Z,0.001,2736 x 26208,5184 x 206352,91584,1200816,2736,5184,12,12
